<a href="https://colab.research.google.com/github/booksandmusic182/hello-world/blob/main/Copy_of_NewRandom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:

%%writefile new_random.cpp

#include <iostream>
#include <cstdlib>
#include <ctime>
int main(int argc, char *argv[]) {
  std::cout << argc << "\n";
   srand(std::stoi(argv[1])); // Seed with current time
   for (int i = 0; i < 5; i++) {
       std::cout << rand() % 100 << " "; // Random number between 0 and 99
   }
   return 0;
}

Overwriting new_random.cpp


In [70]:
%%shell
g++ new_random.cpp -o new_random

In [71]:
!./new_random

1
15 74 65 7 30 

In [6]:
%%writefile random.cpp

 #include <stdio.h>  // printf()
 #include <time.h>   // time()
 #include <string.h> // C++ string comparison

 #include <omp.h>

 // trng YARN (yet another random number) generator class
 #include "new_random.cpp"

 #define LEAPFROG 0
 #define BLOCKSPLIT 1

 ////////////////////////////////////////////////////////////
 int random(int argc, char *argv[])
 {

   // set up what conditions we will use:
   //  - defult size of repetitions in the loop
   //  - whether to use a constant seed so we can repeat and
   //    get same random values in stream

   int repetitions = 8;
   int useConstantSeed = 0; // for same stream, set this to 1 on cammand line with -c

   // method the random generator will use to dole out the numbers
   int doleOut = LEAPFROG;

   // for openMP
   int numThreads = 1;


   // openMP additions +++++++++++++++++++++++++++++++++++++++++++
   int tid = 0;
   omp_set_num_threads(numThreads);
   // Print out info
   printf("trng random number stream will be split ");
   if (doleOut == LEAPFROG)
   {
     printf("using leapfrog.\n");
   }
   else
   {
     printf("into blocks.\n");
   }
   printf("The loop is partitioned into equal chunks per thread.\n");
   printf("This means that the loop indices should be consecutive per thread.\n");
   printf("the output is printed like this:\n");
   printf("thread (loop index):randNumber\n");
   // openMP additions +++++++++++++++++++++++++++++++++++++++++++++

   // ///////////////  random generator setup /////////////////////

   // random numbers start from a seed value
   long unsigned int seedValue; // note for trng this is long unsigned

   // same constant seed will generate the same sequence of rndom numbers
   // use for testing to varify same sequence regardless of number of threads
   if (useConstantSeed)
   {
     seedValue = 888777666;
   }
   else
   {                                            // variable seed based on computer clock time
     seedValue = (long unsigned int)time(NULL); // enables variation; use for simulations
   }

 // For threaded openMP version, each thread will fork and have
 // a private copy of:
 //
 //  its thread id
 //
 // These are also automatically private because they are
 // declared inside the pragma block:
 //  -the generator and the distribution
 //  -the next value that will comback from the generator via the distribution
 //  -the loop counter
 //
 //  Since seedValue and repetitions are read but not written,
 //  they are shared.
 //
 // //////////////   begin fork here by using pragma for the compiler
 #pragma omp parallel default(none) private(tid) \
     shared(seedValue, repetitions, min, max, numThreads, doleOut)
   {
     // number generation needs two things: a generator and a distribution of the numbers
     // declare the generator object
     trng::yarn2 randGen;
     // declare the distribution to use (here it is uniform with vallues in the range min to max)
     trng::uniform_dist<> uniform(min, max);

     // Set the starting point of the generator by seeding it
     randGen.seed(seedValue);

     // OpenMP addition: get my thread number
     tid = omp_get_thread_num();

     // Addition for openMP and parallel in general: the generator must be set to give
     // the thread its portion of the random numbers.
     // Note if single thread, this isn't necessary.
     if (numThreads > 1)
     {
       if (doleOut == LEAPFROG)
       {
         // thread will get substream tid from numThread separate substreams
         randGen.split((unsigned)numThreads, tid); // this is the leapfrog setup
       }
       else
       {
         // thread will get substream as a block:
         // thread 0 starts at 0, thread 1 starts at repetitions / numThreads, etc.
         randGen.jump(tid * (repetitions / numThreads)); // block split
       }
     }
     // //////////////// end PRNG setup /////////////////////////////////

     // //////////////// get a portion of the stream /////////////////////
     int nextRandValue; // holds the next value as we go through the loop

     // loop to get each number in the PRNG stream and print it
     // Note here the ubiquitous for loop construction of incrementing by 1-
     //   this leads itself nicely to letting the openMP compiler split the
     //   loop into equal chunks using the pragma shown.
     int i;
 #pragma omp for
     for (i = 0; i < repetitions; i++)
     {
       // get next number in the stream from the distribution
       nextRandValue = uniform(randGen);
       // print tid(i):nextRandValue
       printf("t%2d (%2d):%2d \n", tid, i, nextRandValue);
     }

   } // end of parallel block

   return 0;
 }


Overwriting random.cpp


In [7]:
%%shell
g++ random.cpp -o random

random.cpp: In function ‘int random(int, char**)’:
random.cpp:88:6: error: ‘trng’ has not been declared
   88 |      trng::yarn2 randGen;
      |      ^~~~
random.cpp:90:6: error: ‘trng’ has not been declared
   90 |      trng::uniform_dist<> uniform(min, max);
      |      ^~~~
random.cpp:90:25: error: expected primary-expression before ‘>’ token
   90 |      trng::uniform_dist<> uniform(min, max);
      |                         ^
random.cpp:90:35: error: ‘min’ was not declared in this scope; did you mean ‘std::min’?
   90 |      trng::uniform_dist<> uniform(min, max);
      |                                   ^~~
      |                                   std::min
In file included from /usr/include/c++/11/bits/char_traits.h:39,
                 from /usr/include/c++/11/ios:40,
                 from /usr/include/c++/11/ostream:38,
                 from /usr/include/c++/11/iostream:39,
                 from new_random.cpp:2,
                 from random.cpp:9:
/usr/include/c++/11/bits/

CalledProcessError: Command 'g++ random.cpp -o random
' returned non-zero exit status 1.